# RSNA Knee Abnormality Detection - Baseline v1

End-to-end pipeline: weak-label extraction from radiology reports, a 2.5D ResNet18 image model with masked multi-label loss, validation on the 58 gold-labeled studies, and `submission.csv` generation.

**Before running:** Settings -> Accelerator -> GPU.

If DICOM decoding fails with an "Unable to decompress" error, run the pip-install cell below first.

## Imports & Config

In [ ]:
import os
import re
import glob
import random
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
TRAIN_SERIES_DIR = os.path.join(DATA_DIR, "train_series")
TEST_SERIES_DIR = os.path.join(DATA_DIR, "test_series")

IMG_SIZE = 224
N_SLICES_PER_SERIES = 3          # -> 3-channel "2.5D" image
MAX_SERIES_PER_STUDY = 3         # full run: 1 sagittal + 1 coronal + 1 axial
BATCH_SIZE = 16
EPOCHS = 8                       # full run
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Quick-test subsample. None = use ALL training studies (full run).
QUICK_TEST_N_TRAIN_STUDIES = None

LABEL_COLS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

print("Device:", DEVICE)

## 1. LOAD METADATA

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
train_series = pd.read_csv(os.path.join(DATA_DIR, "train_series.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
test_series = pd.read_csv(os.path.join(DATA_DIR, "test_series.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

present_labels = [c for c in LABEL_COLS if c in train.columns]
has_gold = train[present_labels].notna().any(axis=1)
print(f"Gold-labeled studies: {has_gold.sum()} / {len(train)}")

## 2. WEAK LABELS FROM REPORTS (reuse the v1 extractor - condensed here)

In [ ]:
NEGATION = [
    r"\bno\b", r"\bnot\b", r"\bwithout\b", r"\bintact\b", r"\bnormal\b",
    r"\bpreserved\b", r"\bunremarkable\b", r"\bnegative for\b",
    r"\bno evidence of\b", r"\bsin\b", r"\bno hay\b", r"\bconservad[ao]s?\b",
    r"\bíntegr[ao]s?\b", r"\bintact[ao]s?\b",
]
LEXICON = {
    "ACL": [r"\bacl\b.{0,40}\b(tear|rupture)\b", r"anterior cruciate ligament.{0,40}(tear|rupture|disrupt)"],
    "MCL": [r"\bmcl\b.{0,40}\b(tear|sprain|injury)\b", r"medial collateral ligament.{0,40}(tear|sprain|injury)"],
    "Medial Meniscus": [r"medial meniscus.{0,60}(tear|torn|extrusion)"],
    "Lateral Meniscus": [r"lateral meniscus.{0,60}(tear|torn|extrusion)"],
    "Medial OA": [r"medial compartment.{0,60}(cartilage loss|chondral|osteoarthrit|degenerat)"],
    "Lateral OA": [r"lateral compartment.{0,60}(cartilage loss|chondral|osteoarthrit|degenerat)"],
    "PF OA": [r"patellofemoral.{0,60}(cartilage loss|chondral|osteoarthrit|degenerat)"],
    "Effusion": [r"\beffusion\b", r"\bderrame articular\b"],
    "Synovitis": [r"\bsynovitis\b", r"\bsinovitis\b"],
    "Baker's": [r"baker'?s? cyst", r"popliteal cyst"],
    "Contusion": [r"bone (bruise|contusion)", r"marrow edema"],
    "Fracture": [r"\bfracture\b", r"\bfractura\b"],
}


def split_sentences(text):
    if not isinstance(text, str):
        return []
    return [c.strip() for c in re.split(r"[\n\r]+|(?<=[.;])\s+(?=[A-ZÁÉÍÓÚÑ])", text) if c.strip()]


def extract_weak_labels(text):
    sentences = split_sentences(text)
    out = {f: np.nan for f in LABEL_COLS}
    for finding, patterns in LEXICON.items():
        mentioned, positive = False, False
        for sent in sentences:
            for p in patterns:
                m = re.search(p, sent, re.IGNORECASE)
                if m:
                    mentioned = True
                    negated = any(re.search(n, sent[: m.start()], re.IGNORECASE) for n in NEGATION)
                    if not negated:
                        positive = True
        if mentioned:
            out[finding] = 1.0 if positive else 0.0
    return out


print("Extracting weak labels from reports (this can take a minute)...")
weak = train["Report"].apply(extract_weak_labels).apply(pd.Series)

# Final label table: gold overrides weak; mask = 1 wherever we have
# either a gold or a weak (extractor-confident) value.
label_table = weak.copy()
mask_table = weak.notna().astype(float)
for c in present_labels:
    gold_vals = train[c]
    override = gold_vals.notna()
    label_table.loc[override, c] = gold_vals[override]
    mask_table.loc[override, c] = 1.0

label_table = label_table.fillna(0.0)  # unknowns filled but MASKED OUT in loss
print("Label table ready:", label_table.shape)

## 3. SERIES SELECTION

In [ ]:
def pick_series_for_study(study_uid, series_df, series_root):
    rows = series_df[series_df["StudyInstanceUID"] == study_uid]
    if rows.empty:
        return []
    chosen = []
    for plane in ["Sagittal", "Coronal", "Axial"]:
        plane_rows = rows[rows["Anatomical_Plane"] == plane]
        if plane_rows.empty:
            continue
        # prefer fluid sensitive series
        plane_rows = plane_rows.sort_values("Fluid_Sensitive", ascending=False)
        series_uid = plane_rows.iloc[0]["SeriesInstanceUID"]
        series_path = os.path.join(series_root, study_uid, series_uid)
        if os.path.isdir(series_path):
            chosen.append(series_path)
        if len(chosen) >= MAX_SERIES_PER_STUDY:
            break
    if not chosen:
        # fallback: just take the first available series
        series_uid = rows.iloc[0]["SeriesInstanceUID"]
        series_path = os.path.join(series_root, study_uid, series_uid)
        if os.path.isdir(series_path):
            chosen.append(series_path)
    return chosen

## 4. DICOM LOADING -> 2.5D image

In [ ]:
def load_series_as_image(series_path, n_slices=N_SLICES_PER_SERIES, size=IMG_SIZE):
    files = sorted(glob.glob(os.path.join(series_path, "*.dcm")))
    if not files:
        return np.zeros((size, size, n_slices), dtype=np.float32)

    idxs = np.linspace(0, len(files) - 1, num=min(n_slices, len(files))).astype(int)
    idxs = list(idxs)
    while len(idxs) < n_slices:  # pad if series has fewer files than n_slices
        idxs.append(idxs[-1])

    slices = []
    for i in idxs:
        try:
            ds = pydicom.dcmread(files[i])
            arr = ds.pixel_array.astype(np.float32)
            # basic normalization
            arr -= arr.min()
            if arr.max() > 0:
                arr /= arr.max()
            slices.append(arr)
        except Exception:
            slices.append(np.zeros((size, size), dtype=np.float32))

    # resize each slice to (size, size)
    resized = []
    for arr in slices:
        t = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)
        t = torch.nn.functional.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)
        resized.append(t.squeeze().numpy())

    img = np.stack(resized, axis=-1)  # (H, W, n_slices)
    return img

## 5. DATASET

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
normalize = T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)


class KneeStudyDataset(Dataset):
    """One (study, series) pair per item; study-level label is shared
    across its series during training. At inference we average
    predictions across a study's series."""

    def __init__(self, study_uids, series_df, series_root, labels_df=None, mask_df=None):
        self.items = []  # list of (study_uid, series_path)
        for uid in study_uids:
            for sp in pick_series_for_study(uid, series_df, series_root):
                self.items.append((uid, sp))
        self.labels_df = labels_df
        self.mask_df = mask_df
        self.study_index = {uid: i for i, uid in enumerate(study_uids)}

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        study_uid, series_path = self.items[idx]
        img = load_series_as_image(series_path)  # (H, W, 3)
        img = torch.from_numpy(img).permute(2, 0, 1).float()  # (3, H, W)
        img = normalize(img)

        if self.labels_df is not None:
            y = torch.tensor(self.labels_df.loc[self.study_index[study_uid]].values, dtype=torch.float32)
            m = torch.tensor(self.mask_df.loc[self.study_index[study_uid]].values, dtype=torch.float32)
            return img, y, m, study_uid
        return img, study_uid

## 6. TRAIN / VAL SPLIT

In [ ]:
all_study_uids = train["StudyInstanceUID"].tolist()
gold_uids = train.loc[has_gold, "StudyInstanceUID"].tolist()
train_uids = train.loc[~has_gold, "StudyInstanceUID"].tolist()

if QUICK_TEST_N_TRAIN_STUDIES is not None and len(train_uids) > QUICK_TEST_N_TRAIN_STUDIES:
    train_uids = random.sample(train_uids, QUICK_TEST_N_TRAIN_STUDIES)
    print(f"[QUICK TEST MODE] Subsampled train_uids down to {len(train_uids)} studies")

label_table = label_table.reset_index(drop=True)
mask_table = mask_table.reset_index(drop=True)

train_ds = KneeStudyDataset(train_uids, train_series, TRAIN_SERIES_DIR, label_table, mask_table)
val_ds = KneeStudyDataset(gold_uids, train_series, TRAIN_SERIES_DIR, label_table, mask_table)

print(f"Train items (study-series pairs): {len(train_ds)}")
print(f"Val items (study-series pairs): {len(val_ds)}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## 7. MODEL

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, len(LABEL_COLS))
model = model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss(reduction="none")
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)


def masked_bce(logits, targets, mask):
    loss = criterion(logits, targets)
    loss = (loss * mask).sum() / mask.sum().clamp(min=1.0)
    return loss

## 8. TRAIN LOOP (keeps the best checkpoint by val AUC, since AUC on

In [ ]:
best_auc = -1.0
best_state = None

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for imgs, ys, ms, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]"):
        imgs, ys, ms = imgs.to(DEVICE), ys.to(DEVICE), ms.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = masked_bce(logits, ys, ms)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    avg_loss = total_loss / len(train_ds)

    # ---- validation on gold studies only ----
    model.eval()
    study_logits = {}
    study_targets = {}
    with torch.no_grad():
        for imgs, ys, ms, uids in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [val]"):
            imgs = imgs.to(DEVICE)
            logits = torch.sigmoid(model(imgs)).cpu().numpy()
            for i, uid in enumerate(uids):
                study_logits.setdefault(uid, []).append(logits[i])
                study_targets[uid] = ys[i].numpy()

    aucs = []
    for j, finding in enumerate(LABEL_COLS):
        y_true, y_pred = [], []
        for uid in study_logits:
            y_true.append(study_targets[uid][j])
            y_pred.append(np.mean([p[j] for p in study_logits[uid]]))
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        if len(np.unique(y_true)) > 1:
            aucs.append(roc_auc_score(y_true, y_pred))
    mean_auc = np.mean(aucs) if aucs else float("nan")

    print(f"Epoch {epoch+1}/{EPOCHS} - train_loss={avg_loss:.4f} - val_mean_AUC(gold)={mean_auc:.4f}")

    if mean_auc > best_auc:
        best_auc = mean_auc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"  -> new best (val_mean_AUC={best_auc:.4f}), checkpoint saved")

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"\nLoaded best checkpoint (val_mean_AUC(gold)={best_auc:.4f}) for inference.")
else:
    print("\nWARNING: no valid AUC computed in any epoch, using last-epoch weights.")

## 9. INFERENCE ON TEST SET -> submission.csv

In [ ]:
test_uids = test["StudyInstanceUID"].tolist()
test_ds = KneeStudyDataset(test_uids, test_series, TEST_SERIES_DIR)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model.eval()
test_preds = {}
with torch.no_grad():
    for imgs, uids in tqdm(test_loader, desc="Inference [test]"):
        imgs = imgs.to(DEVICE)
        probs = torch.sigmoid(model(imgs)).cpu().numpy()
        for i, uid in enumerate(uids):
            test_preds.setdefault(uid, []).append(probs[i])

rows = []
for uid in test_uids:
    if uid in test_preds:
        avg_probs = np.mean(test_preds[uid], axis=0)
    else:
        avg_probs = np.full(len(LABEL_COLS), 0.5)  # fallback if no series found
    rows.append([uid] + list(avg_probs))

submission = pd.DataFrame(rows, columns=["StudyInstanceUID"] + LABEL_COLS)

# make sure column order / names exactly match sample_submission.csv
submission = submission[sample_sub.columns]
submission.to_csv("submission.csv", index=False)
print("\nsubmission.csv written:")
print(submission.head())